In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [2]:
dataset_path = "dataset"

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

In [4]:
train_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

Found 9216 images belonging to 1 classes.


In [5]:
validation_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 2303 images belonging to 1 classes.


In [6]:
num_classes = len(train_generator.class_indices)

print("Classes:", train_generator.class_indices)
print("Number of Classes:", num_classes)

Classes: {'Combined Dataset': 0}
Number of Classes: 1


In [7]:
resnet_base = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

resnet_base.trainable = False

x = resnet_base.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

x = Dropout(0.5)(x)

output = Dense(
    num_classes,
    activation='softmax'
)(x)

resnet_model = Model(
    inputs=resnet_base.input,
    outputs=output
)

In [8]:
resnet_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_resnet = resnet_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5
)

Epoch 1/5


C:\Users\RU-ECE-118-06\anaconda3\lib\site-packages\tensorflow\python\util\dispatch.py:1176: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return dispatch_target(*args, **kwargs)


288/288 [==============================] - 338s 1s/step - loss: 0.0000e+00 - accuracy: 1.0000 - val_loss: 0.0000e+00 - val_accuracy: 1.0000
Epoch 2/5
162/288 [===============>..............] - ETA: 1:56 - loss: 0.0000e+00 - accuracy: 1.0000

In [ ]:
vgg_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

vgg_base.trainable = False

x = vgg_base.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

x = Dropout(0.5)(x)

output = Dense(
    num_classes,
    activation='softmax'
)(x)

vgg_model = Model(
    inputs=vgg_base.input,
    outputs=output
)

In [ ]:
vgg_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_vgg = vgg_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5
)

In [ ]:
validation_generator.reset()

resnet_predictions = resnet_model.predict(
    validation_generator
)

validation_generator.reset()

vgg_predictions = vgg_model.predict(
    validation_generator
)

In [ ]:
ensemble_predictions = (
    resnet_predictions + vgg_predictions
) / 2

In [ ]:
ensemble_classes = np.argmax(
    ensemble_predictions,
    axis=1
)

true_classes = validation_generator.classes

class_names = list(
    validation_generator.class_indices.keys()
)

print("Class Names:")
print(class_names)

In [ ]:
ensemble_accuracy = np.mean(
    ensemble_classes == true_classes
)

print("Ensemble Accuracy:", ensemble_accuracy)

print(
    "Ensemble Accuracy Percentage:",
    ensemble_accuracy * 100
)

In [ ]:
print(
    classification_report(
        true_classes,
        ensemble_classes,
        target_names=class_names
    )
)

In [ ]:
cm = confusion_matrix(
    true_classes,
    ensemble_classes
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")

plt.ylabel("True Label")

plt.title("Ensemble Model Confusion Matrix")

plt.show()

In [ ]:
validation_generator.reset()

resnet_pred = resnet_model.predict(
    validation_generator
)

resnet_classes = np.argmax(
    resnet_pred,
    axis=1
)

resnet_accuracy = np.mean(
    resnet_classes == true_classes
)


validation_generator.reset()

vgg_pred = vgg_model.predict(
    validation_generator
)

vgg_classes = np.argmax(
    vgg_pred,
    axis=1
)

vgg_accuracy = np.mean(
    vgg_classes == true_classes
)


print("ResNet50 Accuracy:", resnet_accuracy * 100)

print("VGG16 Accuracy:", vgg_accuracy * 100)

print("Ensemble Accuracy:", ensemble_accuracy * 100)

In [ ]:
models = [
    "ResNet50",
    "VGG16",
    "Ensemble"
]

accuracies = [
    resnet_accuracy * 100,
    vgg_accuracy * 100,
    ensemble_accuracy * 100
]

plt.figure(figsize=(8, 5))

plt.bar(models, accuracies)

plt.title("Model Accuracy Comparison")

plt.xlabel("Models")

plt.ylabel("Accuracy (%)")

plt.show()

In [ ]:
resnet_model.save("resnet50_model.keras")

vgg_model.save("vgg16_model.keras")

print("Models saved successfully")